# 00. Colab / Kaggle 작업 환경 준비

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jennk2025/HAIKUTHON/blob/main/notebooks/00_colab_setup.ipynb)

이 노트북 하나로 **저장소 코드, Train 데이터 4,000쌍, 스타터 키트**를 한 번에 준비한다.

| 단계 | 내용 |
|---|---|
| 1 | GitHub 저장소를 받는다 (`haiku/` 공용 코드 포함) |
| 2 | 패키지를 설치한다 (`requirements-colab.txt`, 채점 서버 버전에 맞춤) |
| 3 | Drive를 마운트하고 데이터를 준비한다 |

**데이터 보관 방식 (Colab)**
- 처음 실행: Hugging Face에서 받아 `/content/work`에 두고, `MyDrive/HAIKUTHON/haikuthon-train.tar`(약 5GB)로 Drive에 저장한다.
- 다음 세션부터: Drive의 tar를 `/content/work`로 풀어서 쓴다. Drive에서 PNG를 직접 읽으면 느리기 때문이다.
- 실험 결과(JSON)는 `MyDrive/HAIKUTHON/results/`에 저장되어 세션이 끝나도 남는다.
- Drive 용량이 부족하면 `prepare(save_to_drive=False)`로 바꾼다. 매번 HF에서 받는다.

**규정 확인**: 학습은 무료 Colab / Kaggle GPU에서만 한다. 런타임 > 런타임 유형 변경 > **T4 GPU**.

In [ ]:
# 저장소 받기 + 경로 설정 (Colab / Kaggle 공통). 노트북마다 맨 처음 한 번 실행한다.
import os, sys, subprocess
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # GPU 1장만 사용 (채점 서버와 같은 조건). torch import 전에 설정
REPO_URL = "https://github.com/jennk2025/HAIKUTHON.git"
if Path("/kaggle").exists() or Path("/content").exists():
    REPO_DIR = Path("/kaggle/working/HAIKUTHON" if Path("/kaggle").exists() else "/content/HAIKUTHON")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO_DIR)], check=True)
    else:                                    # 이미 받았으면 최신 코드로 갱신
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q"], check=False)
else:                                        # 로컬: notebooks/ 의 상위 폴더 = 저장소
    REPO_DIR = Path.cwd().resolve().parent
sys.path.insert(0, str(REPO_DIR))
print("REPO_DIR =", REPO_DIR)

In [ ]:
%pip install -q -r {REPO_DIR}/requirements-colab.txt

In [ ]:
# 데이터 + 스타터 키트 준비.
# Colab: Drive에 데이터 묶음이 있으면 그걸 풀고, 없으면 HF에서 받은 뒤 Drive에 저장한다(처음 한 번).
# Kaggle: 오른쪽 Settings에서 Internet을 켜야 HF에서 받을 수 있다.
from haiku.setup_data import prepare, env_info
P = prepare(use_drive=True, save_to_drive=True)
DATA_DIR, KIT_DIR, WORK_DIR, RESULTS_DIR = P["data_dir"], P["kit_dir"], P["work_dir"], P["results_dir"]

In [ ]:
import json
ENV_INFO = env_info()
print(json.dumps(ENV_INFO, indent=2, ensure_ascii=False))
assert ENV_INFO.get("cuda_available"), "GPU가 잡히지 않았습니다. 런타임 > 런타임 유형 변경 > GPU(T4)"

## 데이터 확인

In [ ]:
from haiku.data import load_rows, official_split
rows = load_rows(DATA_DIR)
train_rows, valid_rows = official_split(rows, seed=42)
print(f"전체 {len(rows)}쌍 / 공식 분할 학습 {len(train_rows)} · 검증 {len(valid_rows)}")
print("스타터 키트:", sorted(p.name for p in KIT_DIR.iterdir()))

In [ ]:
import random
import matplotlib.pyplot as plt
from PIL import Image
import textwrap

sample = random.Random(0).sample(rows, 3)
fig, axes = plt.subplots(1, 3, figsize=(15, 6))
for ax, r in zip(axes, sample):
    ax.imshow(Image.open(r["image_path"]))
    ax.set_title(r["id"]); ax.axis("off")
    print(f"[{r['id']}] " + textwrap.fill(r["prompt"], 120), "\n")
plt.tight_layout(); plt.show()

## 작업 공간

위 셀을 실행한 뒤 아래에서 자유롭게 작업한다. 공용 코드는 `haiku/`에 있다.

| 모듈 | 내용 |
|---|---|
| `haiku.data` | `load_rows`, `official_split`(스타터 키트와 같은 분할), `to_queries`, `to_candidates`, `gold_of` |
| `haiku.metrics` | `metrics_from_ranks`(공식 점수 + Hit@K + 순위 분포), `score_predictions`(predict 결과 채점) |
| `haiku.retrieval` | `load_siglip`, `embed_images`, `embed_texts`, `ranks_of_gold`, `evaluate_rows` |
| `haiku.timing` | `timed`, `peak_memory_gb`, `test_scale_inputs`, `save_result` |

`haiku/` 코드를 고쳤다면 GitHub에 push한 뒤 첫 셀을 다시 실행하면 최신 코드를 받는다.

In [ ]:
# 여기서 작업